In [ ]:
from google.colab import files

# Upload creditcard.csv from your system
uploaded = files.upload()


In [ ]:
!pip -q install imbalanced-learn


In [ ]:
# Reproducibility
import random
random.seed(1234)

# Core
import os
import numpy as np
import pandas as pd

# Viz
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(rc={'figure.figsize': (12, 8)})

# Stats
from scipy import stats

# Sampling
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import NearMiss, RandomUnderSampler

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics as skm

# Display all columns when printing DataFrames
pd.set_option("display.max_columns", None)


In [ ]:
# If the file isn't found, you'll get an upload prompt
data_path_candidates = ["/content/creditcard.csv", "creditcard.csv"]
for p in data_path_candidates:
    if os.path.exists(p):
        data_path = p
        break
else:
    from google.colab import files
    uploaded = files.upload()   # choose creditcard.csv
    data_path = next(iter(uploaded))  # first uploaded filename

df = pd.read_csv(data_path)
print("Shape:", df.shape)
df.info()


In [ ]:
display(df.describe())
print("\nTarget distribution (Class):")
print(df['Class'].value_counts(normalize=True))


In [ ]:
# Amount stats + distribution (log scale)
print(df['Amount'].describe())
sns.histplot(data=df, x='Amount').set(xscale="log")
plt.xlabel("Dollar Amount (log)")
plt.title("Transaction Amount Distribution (log scale)")
plt.show()

# Correlation heatmap (numeric only, faster without annotations)
corr = df.corr(numeric_only=True).round(2)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, vmax=1, vmin=-1, center=0, cmap='vlag', mask=mask)
plt.title("Correlation Heatmap")
plt.show()

# Pairs with |corr| > 0.5 (excluding 1.0)
cm = df.corr(numeric_only=True).abs().unstack()
cm = cm[cm.between(0.5, 1, inclusive='neither')].sort_values(ascending=False)
print("Correlations with |r| > 0.5:\n", cm.head(10))


In [ ]:
def basic_processing(data, col_drop=None, col_rename=None):
    data = data.copy()
    if col_drop:
        data.drop(col_drop, axis=1, inplace=True)

    if col_rename:
        data.rename(columns=col_rename, inplace=True)

    # Remove duplicates
    before = data.shape[0]
    data.drop_duplicates(keep='first', inplace=True)
    after = data.shape[0]
    if before != after:
        print(f"Removed {before - after} duplicate rows.")
    return data

def imputation_missing_val(data):
    data = data.copy()
    null_cols = data.columns[data.isna().any()].tolist()
    for col in null_cols:
        if data[col].dtype.kind in 'iufc':  # numeric
            data[col] = data[col].fillna(data[col].median())
        else:
            mode_series = data[col].mode()
            fill_value = mode_series.iloc[0] if len(mode_series) else ""
            data[col] = data[col].fillna(fill_value)
    return data

def identify_missing_val(data):
    data = data.copy()
    total_nulls = data.isna().sum().sum()
    print(f"Missing values in dataset: {total_nulls}")
    if total_nulls > 0:
        plt.figure(figsize=(10,6))
        sns.heatmap(data.isna().T, cmap="YlGnBu", cbar_kws={'label':'Missing Data'})
        plt.xlabel("Rows")
        plt.ylabel("Columns")
        plt.show()

        data = imputation_missing_val(data)
        print("After imputation, missing values:", data.isna().sum().sum())
    else:
        print("- No action needed")
    return data

def outliers_detection(data, target_col='target', threshold=3):
    print(f"At ±{threshold} standard deviations:")
    for col in data.columns:
        if col == target_col:
            continue
        z = np.abs(stats.zscore(data[col].astype(float)))
        pos = np.where(z > threshold)[0]
        n = len(pos)
        if n > 0:
            print(f"\t- {col}: {n} outliers")
        else:
            print(f"\t- {col}: no outliers detected")


In [ ]:
df = basic_processing(df, col_drop=[], col_rename={"Class": "target"})
df = identify_missing_val(df)
display(df.head())


In [ ]:
# Boxplot for Amount
sns.boxplot(data=df, x='Amount')
plt.title("Amount Boxplot")
plt.show()

# Outlier counts
outliers_detection(df, target_col='target', threshold=3)

# Amount distributions by class (log scale)
df_no = df[df['target'] == 0]
df_yes = df[df['target'] == 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
fig.suptitle('Amount Distribution by Class (log scale)')
sns.histplot(ax=ax1, data=df_no, x='Amount', stat='probability').set(xscale="log")
ax1.set_title("Non-fraud (target=0)")
sns.histplot(ax=ax2, data=df_yes, x='Amount', stat='probability').set(xscale="log")
ax2.set_title("Fraud (target=1)")
plt.show()


In [ ]:
def class_imbalance_demo(data,
                         sampling_methods={'SMOTE': SMOTE(random_state=42),
                                           'Near-Miss': NearMiss(),
                                           'over': RandomOverSampler(sampling_strategy=0.5, random_state=42),
                                           'under': RandomUnderSampler(sampling_strategy=0.8, random_state=42)}):
    X = data.drop(columns=['target'])
    y = data['target']

    # Individual algorithms
    for name in ['SMOTE', 'Near-Miss']:
        alg = sampling_methods[name]
        X_new, y_new = alg.fit_resample(X, y)
        bal = y_new.value_counts(normalize=True) * 100
        print(f"Shape before {name}: {X.shape} | after: {X_new.shape}")
        print(f"Balance (%):\n{bal}\n")

    # Combined over → under
    X_over, y_over = sampling_methods['over'].fit_resample(X, y)
    X_sampled, y_sampled = sampling_methods['under'].fit_resample(X_over, y_over)
    bal = y_sampled.value_counts(normalize=True) * 100
    print(f"After combined sampling, features: {X_sampled.shape}")
    print(f"Balance (%):\n{bal}\n")

class_imbalance_demo(df)


In [ ]:
# Features / target
X = df.drop(columns=['target'])
y = df['target']

# Hold out an untouched test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

# Resample training only
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE on training:")
print("X_train_res:", X_train_res.shape)
print("y_train_res balance:\n", y_train_res.value_counts(normalize=True))


In [ ]:
def choose_dimred_and_transform(X_tr, X_te, n=5):
    # Fit on training data only
    pca = PCA(n_components=n, random_state=42)
    svd = TruncatedSVD(n_components=n, random_state=42)

    pca.fit(X_tr)
    svd.fit(X_tr)

    pca_var = pca.explained_variance_ratio_.sum()
    svd_var = svd.explained_variance_ratio_.sum()

    best_name = "PCA" if pca_var >= svd_var else "SVD"
    best_model = pca if best_name == "PCA" else svd
    print(f"Dimensionality reduction selected: {best_name} (explained variance {max(pca_var, svd_var):.4f})")

    X_tr_red = best_model.transform(X_tr)
    X_te_red = best_model.transform(X_te)
    return X_tr_red, X_te_red, best_name

# You can set n=5 (as in your code) or try n=10
X_train_red, X_test_red, dimred_name = choose_dimred_and_transform(X_train_res, X_test, n=5)


In [ ]:
def evaluate_models(models, X_tr, y_tr, X_te, y_te, threshold=0.5):
    rows = []
    for name, model in models.items():
        model.fit(X_tr, y_tr)

        # Prefer predict_proba; fallback to decision_function
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_te)[:, 1]
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(X_te)
        else:
            # as a last resort, use predictions (not ideal for AUPRC)
            y_score = model.predict(X_te)

        ap = skm.average_precision_score(y_te, y_score)
        roc = skm.roc_auc_score(y_te, y_score)

        y_pred = (y_score >= threshold).astype(int)
        prec = skm.precision_score(y_te, y_pred, zero_division=0)
        rec  = skm.recall_score(y_te,  y_pred, zero_division=0)
        f1   = skm.f1_score(y_te,     y_pred, zero_division=0)

        rows.append({"Model": name, "AUPRC": ap, "ROC-AUC": roc,
                     "Precision@0.5": prec, "Recall@0.5": rec, "F1@0.5": f1})

    results = pd.DataFrame(rows).sort_values("AUPRC", ascending=False)
    return results

models_simple = {
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=42)
}


In [ ]:
results = evaluate_models(models_simple, X_train_red, y_train_res, X_test_red, y_test)
display(results)
